# Sparsification Percentage — Interactive Study

Explore all analyses interactively. Change the widgets below to filter by model, dataset, solution, and NIID value.

In [ ]:
import importlib.util, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

HERE = Path('.').resolve()
spec = importlib.util.spec_from_file_location('load_data', HERE / 'load_data.py')
ld = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ld)

print('Loading data...')
DFS = ld.load_all()
DIMS = ld.discover_dims(DFS['server_eval'])
print('Done.')
for k, v in DIMS.items():
    print(f'  {k:12s}: {v}')

## Controls

In [ ]:
all_models   = DIMS.get('model', [])
all_datasets = DIMS.get('dataset', [])
all_solutions = DIMS.get('solution', [])
all_niids    = DIMS.get('niid', [])
all_pcts     = [p for p in DIMS.get('percentage', []) if p is not None]
max_round    = int(DFS['server_eval']['round'].max()) if not DFS['server_eval'].empty else 200

w_models    = widgets.SelectMultiple(options=all_models,    value=all_models,    description='Models',    rows=min(5, len(all_models)))
w_datasets  = widgets.SelectMultiple(options=all_datasets,  value=all_datasets,  description='Datasets',  rows=min(5, len(all_datasets)))
w_solutions = widgets.SelectMultiple(options=all_solutions, value=all_solutions, description='Solutions', rows=min(8, len(all_solutions)))
w_niid      = widgets.SelectMultiple(options=all_niids,     value=all_niids,     description='NIID',      rows=min(5, len(all_niids)))
w_metric    = widgets.Dropdown(options=['mean_acc', 'mean_loss'], value='mean_acc', description='Metric')
w_rounds    = widgets.IntRangeSlider(value=[1, max_round], min=1, max=max_round, step=1, description='Rounds', layout=widgets.Layout(width='50%'))
w_export    = widgets.Button(description='Export PNG', button_style='info', icon='download')
w_out_path  = widgets.Text(value='output/figures/notebook_export.png', description='Save to', layout=widgets.Layout(width='60%'))

controls = widgets.HBox([
    widgets.VBox([w_models, w_datasets]),
    widgets.VBox([w_solutions, w_niid]),
    widgets.VBox([w_metric, w_rounds, widgets.HBox([w_export, w_out_path])]),
])
display(controls)

## 1 — Convergence Curves

In [ ]:
out1 = widgets.Output()
display(out1)

def _filter_se(se):
    df = se.copy()
    if 'model' in df.columns:    df = df[df['model'].isin(w_models.value)]
    if 'dataset' in df.columns:  df = df[df['dataset'].isin(w_datasets.value)]
    if 'solution' in df.columns: df = df[df['solution'].isin(w_solutions.value)]
    if 'niid' in df.columns:     df = df[df['niid'].isin(w_niid.value)]
    lo, hi = w_rounds.value
    df = df[(df['round'] >= lo) & (df['round'] <= hi)]
    return df

def _sol_cmap(solutions):
    cmap = plt.cm.get_cmap('tab10', max(len(solutions), 1))
    return {s: cmap(i) for i, s in enumerate(sorted(solutions))}

def update_convergence(*_):
    with out1:
        clear_output(wait=True)
        se = _filter_se(DFS['server_eval'])
        if se.empty:
            print('No data for selected filters.')
            return
        metric = w_metric.value
        models_sel  = [m for m in all_models  if m in (se['model'].unique()  if 'model'  in se.columns else [])]
        niids_sel   = [n for n in all_niids   if n in se['niid'].unique()]
        solutions_sel = [s for s in w_solutions.value if s in se['solution'].unique()]

        n_rows = max(len(models_sel), 1)
        n_cols = max(len(niids_sel),  1)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows), squeeze=False)
        sol_colors = _sol_cmap(solutions_sel)
        styles = ['-', '--', '-.', ':', (0,(3,1,1,1)), (0,(5,2))]
        pcts_sorted = sorted([p for p in se['percentage'].dropna().unique()])
        pct_styles = {p: styles[i % len(styles)] for i, p in enumerate(pcts_sorted)}

        for r, model in enumerate(models_sel or ['']):
            for c, niid in enumerate(niids_sel or ['']):
                ax = axes[r][c]
                sub = se.copy()
                if model and 'model' in sub.columns:  sub = sub[sub['model'] == model]
                if niid  and 'niid'  in sub.columns:  sub = sub[sub['niid']  == niid]
                # plaintext baseline
                pt = sub[sub['percentage'].isna()]
                if not pt.empty and metric in pt.columns:
                    val = pt.groupby('round')[metric].mean().iloc[-1]
                    ax.axhline(val, color='black', linestyle=':', linewidth=1.5, label='plaintext')
                # packing solutions
                pk = sub[sub['percentage'].notna()]
                for sol in solutions_sel:
                    for pct in pcts_sorted:
                        s = pk[(pk['solution']==sol) & (pk['percentage']==pct)].groupby('round')[metric].mean()
                        if s.empty: continue
                        ax.plot(s.index, s.values, color=sol_colors.get(sol,'gray'),
                                linestyle=pct_styles.get(pct,'-'), linewidth=1.2, label=f'{sol} p={pct}')
                ax.set_title(f'{model or ""} | niid={niid or ""}', fontsize=9)
                ax.set_xlabel('Round'); ax.set_ylabel(metric)
                ax.grid(True, alpha=0.3)
                if r == 0 and c == n_cols-1:
                    ax.legend(fontsize=6, loc='lower right', ncol=2)
        fig.suptitle('Convergence Curves', fontsize=12, y=1.01)
        fig.tight_layout()
        plt.show()

for w in [w_models, w_datasets, w_solutions, w_niid, w_metric, w_rounds]:
    w.observe(update_convergence, names='value')

update_convergence()

## 2 — Final Accuracy Heatmap

In [ ]:
out2 = widgets.Output()
display(out2)

def update_heatmap(*_):
    with out2:
        clear_output(wait=True)
        se = _filter_se(DFS['server_eval'])
        packing = se[se['percentage'].notna()]
        if packing.empty:
            print('No packing data.')
            return
        lr = ld.last_round(packing, group_cols=['model','dataset','solution','niid','percentage','seed'])
        summary = lr.groupby(['model','solution','niid','percentage'], dropna=False)['mean_acc'].mean().reset_index()
        models_sel = sorted(summary['model'].unique()) if 'model' in summary.columns else ['']
        niids_sel  = sorted(summary['niid'].unique())
        solutions  = sorted(summary['solution'].unique())
        percentages= sorted(summary['percentage'].dropna().unique())

        n_rows = len(models_sel) * len(niids_sel)
        fig, axes = plt.subplots(n_rows, 1, figsize=(max(6, len(percentages)*1.2), 2.5*n_rows), squeeze=False)
        row = 0
        for model in models_sel:
            for niid in niids_sel:
                ax = axes[row][0]
                sub = summary[(summary['model']==model)&(summary['niid']==niid)] if 'model' in summary.columns else summary[summary['niid']==niid]
                mat = np.full((len(solutions), len(percentages)), np.nan)
                for i, sol in enumerate(solutions):
                    for j, pct in enumerate(percentages):
                        v = sub[(sub['solution']==sol)&(sub['percentage']==pct)]['mean_acc']
                        if not v.empty: mat[i,j] = v.mean()
                im = ax.imshow(mat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
                ax.set_xticks(range(len(percentages))); ax.set_xticklabels([str(p) for p in percentages])
                ax.set_yticks(range(len(solutions)));   ax.set_yticklabels(solutions)
                ax.set_title(f'Accuracy Heatmap — {model} | niid={niid}')
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                for i in range(len(solutions)):
                    for j in range(len(percentages)):
                        val = mat[i,j]
                        if not np.isnan(val):
                            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                                    color='black' if 0.3 < val < 0.8 else 'white')
                row += 1
        fig.tight_layout()
        plt.show()

for w in [w_models, w_datasets, w_solutions, w_niid]:
    w.observe(update_heatmap, names='value')
update_heatmap()

## 3 — HE Overhead

In [ ]:
out3 = widgets.Output()
display(out3)

HE_OPS = ['encrypt_time_s', 'serialize_time_s', 'pack_time_s', 'mask_select_time_s']

def update_overhead(*_):
    with out3:
        clear_output(wait=True)
        tr = DFS['train'].copy()
        if 'model'   in tr.columns: tr = tr[tr['model'].isin(w_models.value)]
        if 'solution'in tr.columns: tr = tr[tr['solution'].isin(w_solutions.value)]
        packing = tr[tr['percentage'].notna()]
        if packing.empty:
            print('No packing train data.')
            return
        present_he = [c for c in HE_OPS if c in packing.columns]
        if not present_he:
            print('HE timing columns not found.')
            return
        packing = packing.copy()
        packing['he_overhead'] = packing[present_he].sum(axis=1)
        models_sel = sorted(packing['model'].unique()) if 'model' in packing.columns else ['']
        solutions  = sorted(packing['solution'].unique())
        cmap = plt.cm.get_cmap('tab10', max(len(solutions),1))

        fig, axes = plt.subplots(len(models_sel), 1, figsize=(7, 4*len(models_sel)), squeeze=False)
        for row, model in enumerate(models_sel):
            ax = axes[row][0]
            sub = packing[packing['model']==model] if 'model' in packing.columns else packing
            agg = sub.groupby(['solution','percentage'], dropna=False)['he_overhead'].mean().reset_index()
            for i, sol in enumerate(solutions):
                s = agg[agg['solution']==sol].sort_values('percentage')
                if s.empty: continue
                ax.plot(s['percentage'], s['he_overhead'], marker='o', label=sol, color=cmap(i), linewidth=1.5)
            ax.set_xlabel('Sparsification percentage')
            ax.set_ylabel('Mean HE overhead (s/round)')
            ax.set_title(f'HE Overhead — {model}')
            ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()

for w in [w_models, w_solutions]:
    w.observe(update_overhead, names='value')
update_overhead()

## 4 — Communication Cost

In [ ]:
out4 = widgets.Output()
display(out4)

def update_comm(*_):
    with out4:
        clear_output(wait=True)
        tr = DFS['train'].copy()
        if 'model'   in tr.columns: tr = tr[tr['model'].isin(w_models.value)]
        if 'solution'in tr.columns: tr = tr[tr['solution'].isin(w_solutions.value)]
        packing = tr[tr['percentage'].notna()]
        if packing.empty or 'payload_bytes' not in packing.columns:
            print('No payload data.')
            return
        models_sel = sorted(packing['model'].unique()) if 'model' in packing.columns else ['']
        solutions  = sorted(packing['solution'].unique())
        cmap = plt.cm.get_cmap('tab10', max(len(solutions),1))

        fig, axes = plt.subplots(len(models_sel), 1, figsize=(7, 4*len(models_sel)), squeeze=False)
        for row, model in enumerate(models_sel):
            ax = axes[row][0]
            sub = packing[packing['model']==model] if 'model' in packing.columns else packing
            agg = sub.groupby(['solution','percentage'], dropna=False)['payload_bytes'].mean().reset_index()
            for i, sol in enumerate(solutions):
                s = agg[agg['solution']==sol].sort_values('percentage')
                if s.empty: continue
                ax.plot(s['percentage'], s['payload_bytes']/1e6, marker='o', label=sol, color=cmap(i), linewidth=1.5)
            ax.set_xlabel('Sparsification percentage')
            ax.set_ylabel('Mean payload (MB)')
            ax.set_title(f'Payload Size — {model}')
            ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()

for w in [w_models, w_solutions]:
    w.observe(update_comm, names='value')
update_comm()

## 5 — NIID Impact

In [ ]:
out5 = widgets.Output()
display(out5)

def update_niid(*_):
    with out5:
        clear_output(wait=True)
        se = DFS['server_eval'].copy()
        if 'model'   in se.columns: se = se[se['model'].isin(w_models.value)]
        if 'solution'in se.columns: se = se[se['solution'].isin(w_solutions.value)]
        packing = se[se['percentage'].notna()]
        if packing.empty:
            print('No packing data.')
            return
        lr = ld.last_round(packing, group_cols=['model','solution','niid','percentage','seed'])
        summary = lr.groupby(['model','solution','niid','percentage'], dropna=False)['mean_acc'].mean().reset_index()
        niids = sorted(summary['niid'].unique())
        if len(niids) < 2:
            print('Need at least 2 NIID values for comparison.')
            return
        solutions  = sorted(summary['solution'].unique())
        percentages= sorted(summary['percentage'].dropna().unique())
        niid_low, niid_high = niids[0], niids[-1]
        models_sel = sorted(summary['model'].unique()) if 'model' in summary.columns else ['']

        fig, axes = plt.subplots(len(models_sel), 1,
                                 figsize=(max(6, len(percentages)*1.2), 2.5*len(models_sel)),
                                 squeeze=False)
        for row, model in enumerate(models_sel):
            ax = axes[row][0]
            sub = summary[summary['model']==model] if 'model' in summary.columns else summary
            mat = np.full((len(solutions), len(percentages)), np.nan)
            for i, sol in enumerate(solutions):
                for j, pct in enumerate(percentages):
                    hi = sub[(sub['solution']==sol)&(sub['niid']==niid_high)&(sub['percentage']==pct)]['mean_acc']
                    lo = sub[(sub['solution']==sol)&(sub['niid']==niid_low) &(sub['percentage']==pct)]['mean_acc']
                    if not hi.empty and not lo.empty: mat[i,j] = hi.mean() - lo.mean()
            vmax = np.nanmax(np.abs(mat)) if not np.all(np.isnan(mat)) else 0.1
            im = ax.imshow(mat, aspect='auto', cmap='RdBu', vmin=-vmax, vmax=vmax)
            ax.set_xticks(range(len(percentages))); ax.set_xticklabels([str(p) for p in percentages])
            ax.set_yticks(range(len(solutions)));   ax.set_yticklabels(solutions)
            ax.set_title(f'Δacc (niid={niid_high}−niid={niid_low}) — {model}')
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            for i in range(len(solutions)):
                for j in range(len(percentages)):
                    val = mat[i,j]
                    if not np.isnan(val): ax.text(j, i, f'{val:+.2f}', ha='center', va='center', fontsize=8)
        fig.tight_layout()
        plt.show()

for w in [w_models, w_solutions]:
    w.observe(update_niid, names='value')
update_niid()

## 6 — Memory Profile

In [ ]:
out6 = widgets.Output()
display(out6)

def update_memory(*_):
    with out6:
        clear_output(wait=True)
        tr = DFS['train'].copy()
        if 'model'   in tr.columns: tr = tr[tr['model'].isin(w_models.value)]
        if 'solution'in tr.columns: tr = tr[tr['solution'].isin(w_solutions.value)]
        if 'ram_peak_mb' not in tr.columns or tr.empty:
            print('No memory data.')
            return
        models_sel = sorted(tr['model'].unique()) if 'model' in tr.columns else ['']
        solutions  = sorted(tr['solution'].unique())
        cmap = plt.cm.get_cmap('tab10', max(len(solutions),1))

        fig, axes = plt.subplots(1, len(models_sel), figsize=(5*len(models_sel), 5), squeeze=False)
        for col, model in enumerate(models_sel):
            ax = axes[0][col]
            sub = tr[tr['model']==model] if 'model' in tr.columns else tr
            data = [sub[sub['solution']==sol]['ram_peak_mb'].dropna().values for sol in solutions]
            bp = ax.boxplot(data, patch_artist=True)
            for patch, i in zip(bp['boxes'], range(len(solutions))):
                patch.set_facecolor(cmap(i)); patch.set_alpha(0.7)
            ax.set_xticks(range(1, len(solutions)+1)); ax.set_xticklabels(solutions, rotation=15, ha='right')
            ax.set_ylabel('Peak RAM (MB)'); ax.set_title(f'Memory — {model}')
            ax.grid(True, axis='y', alpha=0.3)
        fig.tight_layout()
        plt.show()

for w in [w_models, w_solutions]:
    w.observe(update_memory, names='value')
update_memory()

## 7 — Convergence Speed (AUC)

In [ ]:
out7 = widgets.Output()
display(out7)

def update_auc(*_):
    with out7:
        clear_output(wait=True)
        se = _filter_se(DFS['server_eval'])
        packing = se[se['percentage'].notna()]
        if packing.empty:
            print('No packing data.')
            return
        auc_df = ld.auc_accuracy(packing)
        if auc_df.empty or 'auc_norm' not in auc_df.columns:
            print('Could not compute AUC.')
            return
        models_sel = sorted(auc_df['model'].unique()) if 'model' in auc_df.columns else ['']
        solutions  = sorted(auc_df['solution'].unique())
        percentages= sorted(auc_df['percentage'].dropna().unique())
        cmap = plt.cm.get_cmap('tab10', max(len(solutions),1))
        x = np.arange(len(percentages))
        width = 0.8/max(len(solutions),1)

        fig, axes = plt.subplots(len(models_sel), 1, figsize=(8, 4*len(models_sel)), squeeze=False)
        for row, model in enumerate(models_sel):
            ax = axes[row][0]
            sub = auc_df[auc_df['model']==model] if 'model' in auc_df.columns else auc_df
            agg = sub.groupby(['solution','percentage'], dropna=False)['auc_norm'].mean().reset_index()
            for i, sol in enumerate(solutions):
                vals = [agg[(agg['solution']==sol)&(agg['percentage']==p)]['auc_norm'].mean() for p in percentages]
                ax.bar(x + i*width, vals, width, label=sol, color=cmap(i), alpha=0.85)
            ax.set_xticks(x + width*(len(solutions)-1)/2)
            ax.set_xticklabels([str(p) for p in percentages])
            ax.set_xlabel('Sparsification percentage')
            ax.set_ylabel('Normalised AUC')
            ax.set_title(f'Accuracy AUC — {model}')
            ax.legend(fontsize=7); ax.grid(True, axis='y', alpha=0.3)
        fig.tight_layout()
        plt.show()

for w in [w_models, w_solutions, w_niid, w_rounds]:
    w.observe(update_auc, names='value')
update_auc()

## 8 — Efficiency Frontier (Pareto)

In [ ]:
out8 = widgets.Output()
display(out8)

def _pareto_front(costs, accs):
    n = len(costs)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j: continue
            if costs[j] <= costs[i] and accs[j] >= accs[i]:
                if costs[j] < costs[i] or accs[j] > accs[i]:
                    dominated[i] = True; break
    return ~dominated

def update_pareto(*_):
    with out8:
        clear_output(wait=True)
        tr = DFS['train'].copy()
        se = _filter_se(DFS['server_eval'])
        if 'model'   in tr.columns: tr = tr[tr['model'].isin(w_models.value)]
        if 'solution'in tr.columns: tr = tr[tr['solution'].isin(w_solutions.value)]
        if 'payload_bytes' not in tr.columns or tr.empty or se.empty:
            print('No payload data.')
            return
        packing_tr = tr[tr['percentage'].notna()]
        packing_se = se[se['percentage'].notna()]
        payload_agg = packing_tr.groupby(['model','solution','niid','percentage'], dropna=False)['payload_bytes'].mean().reset_index()
        payload_agg['payload_mb'] = payload_agg['payload_bytes']/1e6
        lr  = ld.last_round(packing_se)
        acc_agg = lr.groupby(['model','solution','niid','percentage'], dropna=False)['mean_acc'].mean().reset_index()
        merged = acc_agg.merge(payload_agg, on=['model','solution','niid','percentage'], how='inner')
        if merged.empty:
            print('No merged data.')
            return
        solutions  = sorted(merged['solution'].unique())
        percentages= sorted(merged['percentage'].dropna().unique())
        niids      = sorted(merged['niid'].unique())
        models_sel = sorted(merged['model'].unique()) if 'model' in merged.columns else ['']
        cmap = plt.cm.get_cmap('tab10', max(len(solutions),1))
        sol_colors = {s: cmap(i) for i, s in enumerate(solutions)}
        markers = ['o','s','^','D','v','P','*','X']
        pct_markers = {p: markers[i%len(markers)] for i, p in enumerate(percentages)}

        fig, axes = plt.subplots(len(niids), len(models_sel),
                                 figsize=(6*len(models_sel), 5*len(niids)), squeeze=False)
        for row, niid in enumerate(niids):
            for col, model in enumerate(models_sel):
                ax = axes[row][col]
                sub = merged[(merged['model']==model)&(merged['niid']==niid)] if 'model' in merged.columns else merged[merged['niid']==niid]
                if sub.empty: ax.set_visible(False); continue
                pareto = _pareto_front(sub['payload_mb'].values, sub['mean_acc'].values)
                for i, sol in enumerate(solutions):
                    s = sub[sub['solution']==sol]
                    for _, r in s.iterrows():
                        ax.scatter(r['payload_mb'], r['mean_acc'], color=sol_colors[sol],
                                   marker=pct_markers.get(r['percentage'],'o'), s=90, zorder=3, alpha=0.85)
                pt_pts = sub[pareto].sort_values('payload_mb')
                if not pt_pts.empty:
                    ax.plot(pt_pts['payload_mb'], pt_pts['mean_acc'], 'k--', linewidth=1.5, label='Pareto')
                    ax.scatter(pt_pts['payload_mb'], pt_pts['mean_acc'], color='black', s=50, marker='*', zorder=5)
                for sol in solutions:
                    ax.scatter([],[], color=sol_colors[sol], label=sol, s=60)
                for pct in percentages:
                    ax.scatter([],[], color='gray', marker=pct_markers[pct], label=f'p={pct}', s=50)
                ax.set_xlabel('Mean payload (MB)'); ax.set_ylabel('Final accuracy')
                ax.set_title(f'Pareto — {model} | niid={niid}')
                ax.legend(fontsize=6, ncol=2, loc='lower right'); ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()

for w in [w_models, w_solutions, w_niid]:
    w.observe(update_pareto, names='value')
update_pareto()

## 9 — Summary Table

In [ ]:
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location('_st', HERE / '09_summary_table.py')
_st = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_st)

summary = _st.build_summary(DFS)
display(summary.style
    .format(precision=4)
    .highlight_max(subset=[c for c in ['final_acc','auc_acc'] if c in summary.columns], color='lightgreen')
    .highlight_min(subset=[c for c in ['final_loss','mean_payload_mb','mean_he_overhead_s'] if c in summary.columns], color='lightyellow')
)

## Export current convergence plot

In [ ]:
export_out = widgets.Output()
display(export_out)

def _on_export(_):
    with export_out:
        clear_output(wait=True)
        se = _filter_se(DFS['server_eval'])
        metric = w_metric.value
        solutions_sel = list(w_solutions.value)
        models_sel = list(w_models.value)
        niids_sel  = [n for n in all_niids if n in se['niid'].unique()]
        sol_colors = _sol_cmap(solutions_sel)
        styles = ['-','--','-.',':',(0,(3,1,1,1)),(0,(5,2))]
        pcts_sorted = sorted([p for p in se['percentage'].dropna().unique()])
        pct_styles = {p: styles[i%len(styles)] for i, p in enumerate(pcts_sorted)}
        n_rows = max(len(models_sel),1)
        n_cols = max(len(niids_sel),1)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows), squeeze=False)
        for r, model in enumerate(models_sel or ['']):
            for c, niid in enumerate(niids_sel or ['']):
                ax = axes[r][c]
                sub = se.copy()
                if model and 'model' in sub.columns: sub = sub[sub['model']==model]
                if niid  and 'niid'  in sub.columns: sub = sub[sub['niid']==niid]
                pt = sub[sub['percentage'].isna()]
                if not pt.empty and metric in pt.columns:
                    val = pt.groupby('round')[metric].mean().iloc[-1]
                    ax.axhline(val, color='black', linestyle=':', linewidth=1.5, label='plaintext')
                pk = sub[sub['percentage'].notna()]
                for sol in solutions_sel:
                    for pct in pcts_sorted:
                        s = pk[(pk['solution']==sol)&(pk['percentage']==pct)].groupby('round')[metric].mean()
                        if s.empty: continue
                        ax.plot(s.index, s.values, color=sol_colors.get(sol,'gray'),
                                linestyle=pct_styles.get(pct,'-'), linewidth=1.2, label=f'{sol} p={pct}')
                ax.set_title(f'{model} | niid={niid}', fontsize=9)
                ax.set_xlabel('Round'); ax.set_ylabel(metric)
                ax.grid(True, alpha=0.3)
                ax.legend(fontsize=6, loc='lower right', ncol=2)
        fig.tight_layout()
        out_path = Path(w_out_path.value)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        print(f'Exported to {out_path.resolve()}')
        plt.close(fig)

w_export.on_click(_on_export)
print('Click "Export PNG" to save the current convergence plot.')